# D3 · Which point guard should the club buy?

## The question

> A club wants a strong start next season. It believes that if it invests in a
> position that typically demands high basketball IQ and buys a capable player
> for that position, it can increase its chances of success next season. It
> therefore wants to buy a **Point Guard** with high ability. For this club, the
> ability metric is **presence on the Michael Jordan Trophy list**, and a player
> with more appearances has higher priority. Using season stats from 2019-20
> through the end of 2023-24, produce a list of suitable players to buy and
> present **3 recommendations**.

## Reading the question

Four things need pinning down, and the fourth is not in the brief at all.

**"The Michael Jordan Trophy list."** The trophy is the NBA's Most Valuable
Player award, renamed after Jordan in 2022. One player wins it a year, so a list
of winners across five seasons is five names and nobody has "more appearances"
than anybody else. The MVP *ballot* is the list the question needs. About a
hundred voters each rank five players, everyone who collects a vote is
published, and that comes to between 9 and 15 names a season. It is the reading
used across this project.

**"Point guard."** The database labels position twice. `primary_position` is the
career label from a player's bio page. `position` is what he was listed at in
the season being described. This analysis uses the season label, so a player
counts only for the seasons he actually did the job.

The two labels are not interchangeable. Of the 697 players with more than one
season inside this window, 226 were listed at a different position in different
seasons, and 62 played point guard in some seasons and something else in others.
LeBron James is the case that decides it here. His career label is small forward,
and he was listed at point guard in both of the seasons he made the ballot inside
this window. The career reading throws him out of the pool. The season reading
keeps him, which is right, because in those seasons he was the one bringing the
ball up.

**The window.** Seasons are stored under their ending year, so 2019-20 through
2023-24 is 2020 to 2024 inclusive.

**The tie-break, which the brief does not supply.** "More appearances has higher
priority" orders the pool until two players have the same count, and then it
stops. Three names cannot be pulled out of a count on its own. So the rule is
fixed here, before the data is looked at:

1. more ballot appearances as a point guard, then
2. the better average finish in the vote (`mvp_rank`, where 1 is the winner),
   then
3. alphabetically by name.

Nothing later in the notebook reorders that list.

One thing to say before starting. The client's metric only looks backwards. It
counts how often a player was admired across five past seasons and says nothing
about how old he is, whether he is fit, or whether he is still good. The brief
asks for that metric, so it gets served first and tested second.

## What we need, and where it comes from

For each of the five seasons from 2019-20 to 2023-24: every player who received
at least one MVP vote, the position he played that season, and where he finished
in the voting. That is enough to build the list the client asked for.

Judging the list needs a little more about the same players. How old each one
was, how many of his club's games he was fit to play, how much he scored, how
much of his team's passing ran through him, and the usual all-in-one summaries of
how well he played.

One piece comes from outside the window. The club is buying for the season
straight after it ends, and the database now holds that season and the one after
it. Neither was available to the club at the time, so neither is used to pick
anybody. They are here to grade the metric afterwards.

All of it sits in a single table with one row per player per season, so nothing
has to be joined.

In [1]:
import _setup  # noqa: F401

import pandas as pd

from utils.custom_plots import bubble_plot, dumbbell_plot, radar_plot
from utils.custom_stats import correlation_test
from utils.db_utils import run_query

In [2]:
# Both position labels are pulled, so the choice between them can be shown
# rather than asserted.
SQL_POOL = """
SELECT season,
       season_label,
       player_id,
       player_name,
       position,
       primary_position,
       age,
       mvp_rank,
       mvp_vote_share,
       player_efficiency_rate
FROM analyst_ready.player_season
WHERE season BETWEEN 2020 AND 2024
  AND is_mvp_candidate
  AND (position = 'PG' OR primary_position = 'PG')
ORDER BY player_name, season
"""

pool = run_query(SQL_POOL)

# SQL `numeric` arrives as Decimal typed `object`, and several plotting helpers
# skip object columns without saying so. Cast once, here.
pool[["mvp_vote_share", "player_efficiency_rate"]] = (
    pool[["mvp_vote_share", "player_efficiency_rate"]].astype(float)
)

print(pool.shape)
pool.head()

(27, 10)


,season,season_label,player_id,player_name,position,primary_position,age,mvp_rank,mvp_vote_share,player_efficiency_rate
0,2021,2020-21,simmobe01,Ben Simmons,PG,PG,24,12,0.003,18.3
1,2020,2019-20,paulch01,Chris Paul,PG,PG,34,7,0.026,21.7
2,2021,2020-21,paulch01,Chris Paul,PG,PG,35,5,0.138,21.4
3,2022,2021-22,paulch01,Chris Paul,PG,PG,36,9,0.002,20.8
4,2020,2019-20,lillada01,Damian Lillard,PG,PG,29,8,0.023,26.9


In [3]:
SQL_SCALE = """
SELECT count(DISTINCT player_id) FILTER (WHERE position = 'PG')  AS point_guards,
       count(*)                  FILTER (WHERE is_mvp_candidate) AS ballot_rows,
       count(DISTINCT player_id) FILTER (WHERE is_mvp_candidate) AS ballot_players,
       count(*)                  FILTER (WHERE is_mvp_candidate
                                          AND position = 'PG')   AS pg_ballot_rows,
       count(DISTINCT player_id) FILTER (WHERE is_mvp_candidate
                                          AND position = 'PG')   AS pg_ballot_players
FROM analyst_ready.player_season
WHERE season BETWEEN 2020 AND 2024
"""

scale = run_query(SQL_SCALE)
print(scale.T.rename(columns={0: "count"}).to_string(), end="\n\n")

print(f"{scale.at[0, 'pg_ballot_players']} of {scale.at[0, 'point_guards']} point "
      f"guards ({scale.at[0, 'pg_ballot_players'] / scale.at[0, 'point_guards']:.1%}) "
      f"drew an MVP vote in this window")
print(f"point guards take {scale.at[0, 'pg_ballot_rows']} of the "
      f"{scale.at[0, 'ballot_rows']} ballot places "
      f"({scale.at[0, 'pg_ballot_rows'] / scale.at[0, 'ballot_rows']:.0%})")

                   count
point_guards         200
ballot_rows           61
ballot_players        29
pg_ballot_rows        26
pg_ballot_players     13

13 of 200 point guards (6.5%) drew an MVP vote in this window
point guards take 26 of the 61 ballot places (43%)


Thirteen names, and that is the whole shopping list.

200 different point guards played at least one season in this window. Thirteen of
them ever drew an MVP vote, which is 6.5% of the position. The other 93.5% are
invisible to the client's metric, whatever they did on a court.

The position is not being short-changed. Its 26 ballot places are 43% of all 61
places in the window, so point guards are well represented among the players
voters notice. There are simply very few MVP candidates. A metric with an intake
of nine to fifteen names a year was never going to produce a long list, and a
club that shops this way has ruled out 94% of the market before it starts.

In [4]:
pg_seasons = pool[pool["position"] == "PG"]

reading = (
    pool.assign(
        season_pg=(pool["position"] == "PG").astype(int),
        career_pg=(pool["primary_position"] == "PG").astype(int),
    )
    .groupby("player_name", as_index=False)[["season_pg", "career_pg"]]
    .sum()
)

print("candidates under the season-position reading:",
      int((reading["season_pg"] > 0).sum()))
print("candidates under the career-position reading:",
      int((reading["career_pg"] > 0).sum()), end="\n\n")

print("where the two readings disagree, in ballot seasons counted:")
print(reading.loc[reading["season_pg"] != reading["career_pg"]].to_string(index=False),
      end="\n\n")

print("the ballot seasons a player spent away from his career label:")
print(pool.loc[pool["position"] != pool["primary_position"],
               ["season_label", "player_name", "position", "primary_position",
                "mvp_rank", "mvp_vote_share"]].to_string(index=False))

candidates under the season-position reading: 13
candidates under the career-position reading: 12

where the two readings disagree, in ballot seasons counted:
 player_name  season_pg  career_pg
James Harden          1          2
LeBron James          2          0

the ballot seasons a player spent away from his career label:
season_label  player_name position primary_position  mvp_rank  mvp_vote_share
     2019-20 James Harden       SG               PG         3           0.363
     2019-20 LeBron James       PG               SF         2           0.746
     2020-21 LeBron James       PG               SF        13           0.001


Two players move, and they move in opposite directions.

James Harden is a point guard by career label and made two ballots inside the
window, but the better of the two, third place in 2019-20, he played at shooting
guard. Under the season reading that appearance does not count towards buying a
point guard. LeBron James is the mirror image: small forward by career label,
listed at point guard for both of his ballot seasons here.

So the season reading gives 13 candidates and the career reading gives 12, and
they are not the same 12 with one added. The top three come out identical either
way, because nobody near the top of the list changed position. The choice moves
the shape of the pool rather than the answer, and it is still the right one. The
club is buying somebody to run its offence, and the question is whether he ran
one.

In [5]:
ranked = (
    pg_seasons.groupby(["player_id", "player_name"], as_index=False)
    .agg(
        ballot_seasons=("season", "size"),
        mean_ballot_rank=("mvp_rank", "mean"),
        best_ballot_rank=("mvp_rank", "min"),
        total_vote_share=("mvp_vote_share", "sum"),
        last_ballot_season=("season", "max"),
        age_last_ballot=("age", "max"),
        per_on_ballot=("player_efficiency_rate", "mean"),
    )
    # The tie-break declared above: appearances, then average finish, then name.
    .sort_values(["ballot_seasons", "mean_ballot_rank", "player_name"],
                 ascending=[False, True, True])
    .reset_index(drop=True)
)
ranked.index += 1

# Three age bands, so the chart below can colour by something a reader can read.
ranked["age_band"] = pd.cut(
    ranked["age_last_ballot"], bins=[0, 24, 29, 99],
    labels=["24 and under", "25-29", "30 and over"],
).astype(str)

ranked[["player_name", "ballot_seasons", "mean_ballot_rank", "best_ballot_rank",
        "total_vote_share", "last_ballot_season", "age_last_ballot"]].round(2)

,player_name,ballot_seasons,mean_ballot_rank,best_ballot_rank,total_vote_share,last_ballot_season,age_last_ballot
1,Luka Dončić,5,5.20,3,0.97,2024,24
2,Stephen Curry,3,6.67,3,0.46,2023,34
3,Chris Paul,3,7.00,5,0.17,2022,36
4,Shai Gilgeous-Alexander,2,3.50,2,0.69,2024,25
5,Damian Lillard,2,7.50,7,0.06,2021,30
6,LeBron James,2,7.50,2,0.75,2021,36
7,Jalen Brunson,2,8.50,5,0.14,2024,27
8,Ja Morant,2,9.50,7,0.01,2023,23
9,Derrick Rose,1,9.00,9,0.01,2021,32
10,De'Aaron Fox,1,11.00,11,0.00,2023,25


## The list the client asked for

Applied exactly as written, the metric returns **Luka Dončić**, **Stephen Curry**
and **Chris Paul**.

The tie-break earns its keep in the middle of the list rather than at the top.
Membership of the top three is settled by appearances alone, because one player
has five and exactly two have three. Which of those two comes second needs the
second rule: Curry averages 6.67 in the vote against Paul's 7.00. Further down,
Damian Lillard and LeBron James both have two appearances and both average
exactly 7.5, so the alphabetical rule fires and separates them.

That is the answer to the question as asked. The rest of the notebook is about
whether it is an answer worth acting on.

## What one appearance is actually worth

Being on the ballot is a yes or no fact, and it hides an enormous range.
`mvp_rank` says where a player finished. `mvp_vote_share` says what proportion of
the available voting points he actually won, and it is the finer instrument. A
player who finished thirteenth on one stray fifth-place vote and a player who
finished second with half the panel behind him each count as one appearance.

Chris Paul's three came at 7th, 5th and 9th, worth 0.026, 0.138 and 0.002 of the
vote. Shai Gilgeous-Alexander's two came at 5th and 2nd, worth 0.046 and 0.646.
Two of the second kind outweigh three of the first by a factor of four.

In [6]:
correlation_test(
    ranked, pairs=[("ballot_seasons", "total_vote_share")], method="spearman"
).T

,0
x,ballot_seasons
y,total_vote_share
method,spearman
n,13
r,0.86034
ci_low,0.576052
ci_high,0.958957
r_squared,0.740185
p_value,0.000161
p_adj,0.000161


Across all thirteen the two orderings agree more than that example suggests.
Spearman's rho between appearance count and total vote share is 0.86, with a 95%
interval from 0.58 to 0.96 and p = 0.0002. The routine flags the obvious caveat
in its `flags` column: n = 13, so the interval is wide and unstable. As a way of
sorting thirteen players into roughly the right order, counting appearances
works.

It breaks at the top, which is the only part a club with three signings to make
cares about. By total vote share the order runs Dončić 0.97, LeBron James 0.75,
Gilgeous-Alexander 0.69, Curry 0.46, Paul 0.17. The client's metric puts Paul
third. The votes he actually won put him fifth, behind two players with fewer
appearances than him.

In [7]:
bubble_plot(
    ranked,
    x="mean_ballot_rank",
    y="total_vote_share",
    size="ballot_seasons",
    color_by="age_band",
    label_col="player_name",
    label_top_n=len(ranked),
    log_y=True,
    size_range=(11.0, 42.0),
    title="Every point guard who drew an MVP vote, 2019-20 to 2023-24",
)

Bubble area is the client's metric. Left is a better average finish. Height is
total vote share on a log axis, which the pool needs, because it runs from 0.001
to 0.97. Colour is the player's age in the last season he made the ballot.

Three things are visible at once. Dončić is top left with the biggest bubble,
which is where a club wants its target to be. Chris Paul's bubble is the same
size as Curry's and sits a full order of magnitude below it. Gilgeous-Alexander
is the small bubble furthest left and third from the top, and he was 25 the last
time voters put him on a ballot.

Seven of the thirteen were 30 or older by their final appearance. That is not an
accident of this sample. Reputation takes years to build, so a metric made of
reputation will keep pointing at players who are past the age a club would rather
buy them at.

## Is any of them still good?

The club decides in the summer of 2024 and wants a strong 2024-25. Its metric
looks at 2019-20 through 2023-24, and none of it is about next season.

There is a check for that which needs no hindsight at all. All thirteen played in
2023-24, the last season inside the window, whether or not they drew a vote for
it. That season is the most recent evidence the club has, and the metric ignores
it unless it came with votes attached.

Player Efficiency Rating is the natural yardstick here, because it is rescaled
every year so that the league average is exactly 15.0. A 21 from 2020-21 and a 21
from 2023-24 mean the same thing, which is what makes reputation and current form
comparable on one axis.

In [8]:
# 2024 is the last season the club can see. 2025 and 2026 come later, and are
# used only to grade the metric after the recommendation has been made.
SQL_LATEST = """
WITH candidates AS (
    SELECT DISTINCT player_id
    FROM analyst_ready.player_season
    WHERE season BETWEEN 2020 AND 2024
      AND is_mvp_candidate
      AND position = 'PG'
)
SELECT ps.season,
       ps.season_label,
       ps.player_id,
       ps.player_name,
       ps.age,
       ps.games_played,
       ps.availability,
       ps.points_per_game,
       ps.assist_percentage,
       ps.true_shooting_percentage,
       ps.player_efficiency_rate,
       ps.value_over_replacement_player,
       ps.mvp_rank
FROM analyst_ready.player_season AS ps
JOIN candidates USING (player_id)
WHERE ps.season BETWEEN 2024 AND 2026
ORDER BY ps.season, ps.value_over_replacement_player DESC
"""

latest = run_query(SQL_LATEST)
LATEST_NUMERIC = ["availability", "points_per_game", "assist_percentage",
                  "true_shooting_percentage", "player_efficiency_rate",
                  "value_over_replacement_player"]
latest[LATEST_NUMERIC] = latest[LATEST_NUMERIC].astype(float)

print(latest.shape)
print(latest.groupby("season_label")["player_id"].nunique()
      .rename("candidates who played").to_string())

(35, 13)
season_label
2023-24    13
2024-25    12
2025-26    10


In [9]:
last_seen = latest[latest["season"] == 2024].set_index("player_id")

form = ranked.copy()
for new_col, source_col in (("age_2023_24", "age"),
                            ("games_2023_24", "games_played"),
                            ("per_2023_24", "player_efficiency_rate")):
    form[new_col] = last_seen[source_col].reindex(form["player_id"]).to_numpy()
form["seasons_since_ballot"] = 2024 - form["last_ballot_season"]

form[["player_name", "ballot_seasons", "mean_ballot_rank", "total_vote_share",
      "seasons_since_ballot", "age_2023_24", "games_2023_24",
      "per_on_ballot", "per_2023_24"]].round(2)

,player_name,ballot_seasons,mean_ballot_rank,total_vote_share,seasons_since_ballot,age_2023_24,games_2023_24,per_on_ballot,per_2023_24
1,Luka Dončić,5,5.20,0.97,0,24,70,26.96,28.1
2,Stephen Curry,3,6.67,0.46,1,35,74,23.93,20.6
3,Chris Paul,3,7.00,0.17,2,38,58,21.30,14.7
4,Shai Gilgeous-Alexander,2,3.50,0.69,0,25,75,28.25,29.3
5,Damian Lillard,2,7.50,0.06,3,33,73,26.25,19.6
6,LeBron James,2,7.50,0.75,3,39,71,24.85,23.7
7,Jalen Brunson,2,8.50,0.14,0,27,77,22.30,23.4
8,Ja Morant,2,9.50,0.01,1,24,9,23.85,20.6
9,Derrick Rose,1,9.00,0.01,3,35,24,18.30,13.1
10,De'Aaron Fox,1,11.00,0.00,1,26,74,21.80,20.1


In [10]:
dumbbell_plot(
    form,
    category_col="player_name",
    value_col_start="per_on_ballot",
    value_col_end="per_2023_24",
    sort_by="end",
    ascending=False,
    start_label="average PER across his ballot seasons",
    end_label="PER in 2023-24, the last season the club can see",
    title="Reputation against current form, at the moment the club decides",
    height=640,
)

Ten of the thirteen declined, which on its own is unremarkable. A player reaches
an MVP ballot in his best seasons, so any later season is being measured against
his own peak.

The size of the drop is what separates them, and two names fall through the
floor. Chris Paul lost 6.6 points of PER, from 21.3 across his ballot seasons to
14.7 in 2023-24, which is below the league average of 15.0. He was 38, played 58
games and scored 9.2 a game. Derrick Rose is worse at 13.1, from 24 games at the
age of 35. Both are on the client's list, and Paul is third on it.

The recency column in the table above says the same thing more bluntly. Paul's
last point-guard ballot appearance was 2021-22, already two seasons stale by the
time the club is buying. Rose's was 2020-21, and so were the last appearances of
five other candidates. The metric counts appearances and never asks when they
happened.

Three players improved, all by 1.1 points of PER: Gilgeous-Alexander, Dončić and
Jalen Brunson. They are also the only three whose most recent ballot appearance
is 2023-24, the season that has just finished.

In [11]:
RADAR_METRICS = ["points_per_game", "assist_percentage",
                 "true_shooting_percentage", "player_efficiency_rate",
                 "value_over_replacement_player", "availability"]

radar_plot(
    latest[latest["season"] == 2024],
    category_col="player_name",
    value_cols=RADAR_METRICS,
    entities=["Luka Dončić", "Stephen Curry", "Chris Paul",
              "Shai Gilgeous-Alexander"],
    title="The three the metric picks, and the one it ranks fourth, in 2023-24",
    height=680,
)

Each axis is scaled across all thirteen candidates, so a polygon reaching the
outer ring means best in the pool that season, not best of these four.

Chris Paul has one thing left, and it is passing. His assist percentage is 33.4
against Curry's 24.7, so he was still setting up more of his team's baskets than
the man ranked above him. On scoring, shooting efficiency and PER he is second or
third from the bottom of the thirteen. Nine points a game on a true shooting
percentage of 0.544 is a backup's line, and he missed 24 games on top of it.

Gilgeous-Alexander and Dončić sit outside Curry on every axis except
availability, where Curry's 74 games edges Dončić's 70. Dončić owns assist
percentage at 44.3, the highest in the pool and the most point-guard-specific
number on the chart. Gilgeous-Alexander leads on scoring, shooting efficiency and
PER, and played 75 games doing it.

## What happened next

Everything above uses only what the club had in the summer of 2024. The two
seasons below came later, and they are here to grade the metric rather than to
pick anybody.

In [12]:
epilogue = pd.concat(
    {label: group.set_index("player_name")[
        ["games_played", "player_efficiency_rate", "mvp_rank"]]
     for label, group in latest[latest["season"] > 2024].groupby("season_label")},
    axis=1,
).reindex(form["player_name"])

# A blank row means the player did not appear in the league that season.
epilogue

2024-25                                  \
                        games_played player_efficiency_rate mvp_rank   
player_name                                                            
Luka Dončić                     50.0                   24.1      NaN   
Stephen Curry                   70.0                   21.5      9.0   
Chris Paul                      82.0                   14.7      NaN   
Shai Gilgeous-Alexander         76.0                   30.7      1.0   
Damian Lillard                  58.0                   21.3      NaN   
LeBron James                    70.0                   22.7      6.0   
Jalen Brunson                   65.0                   21.6     10.0   
Ja Morant                       50.0                   19.3      NaN   
Derrick Rose                     NaN                    NaN      NaN   
De'Aaron Fox                    62.0                   18.3      NaN   
Russell Westbrook               75.0                   14.3      NaN   
Ben Simmons                     51.0                   12.4      NaN   
James Harden                    79.0                   20.0     10.0   

                             2025-26                                  
                        games_played player_efficiency_rate mvp_rank  
player_name                                                           
Luka Dončić                     64.0                   27.9      4.0  
Stephen Curry                   43.0                   22.6      NaN  
Chris Paul                      16.0                    8.1      NaN  
Shai Gilgeous-Alexander         68.0                   30.8      1.0  
Damian Lillard                   NaN                    NaN      NaN  
LeBron James                    60.0                   20.8      NaN  
Jalen Brunson                   74.0                   20.1      NaN  
Ja Morant                       20.0                   16.6      NaN  
Derrick Rose                     NaN                    NaN      NaN  
De'Aaron Fox                    72.0                   17.6      NaN  
Russell Westbrook               64.0                   15.1      NaN  
Ben Simmons                      NaN                    NaN      NaN  
James Harden                    70.0                   20.7      NaN

Gilgeous-Alexander won the award in 2024-25 and again in 2025-26, at a PER of
30.7 and 30.8. Dončić stayed excellent per minute and kept missing games, 50 then
64. Curry held up for one season at 21.5 and then played 43 games. Chris Paul
played all 82 games in 2024-25 at a PER of 14.7, then 16 games at 8.1. Derrick
Rose does not appear at all, because he retired.

So the metric's first two picks were fine and its third was a 39-year-old
backup. The player it ranked fourth won the MVP twice.

## Conclusion

**Applied exactly as written, the client's metric returns Luka Dončić, Stephen
Curry and Chris Paul. Two of those are worth buying. The third had already
dropped below the league average in the last season of the window, and the metric
cannot see it, because the metric only looks at seasons that came with votes
attached.**

The five names the recommendation turns on:

| | ballot seasons | mean finish | total vote share | last ballot | age in 2023-24 | PER in 2023-24 |
| --- | ---: | ---: | ---: | :---: | ---: | ---: |
| Luka Dončić | 5 | 5.20 | 0.97 | 2023-24 | 24 | 28.1 |
| Stephen Curry | 3 | 6.67 | 0.46 | 2022-23 | 35 | 20.6 |
| Chris Paul | 3 | 7.00 | 0.17 | 2021-22 | 38 | 14.7 |
| Shai Gilgeous-Alexander | 2 | 3.50 | 0.69 | 2023-24 | 25 | 29.3 |
| Jalen Brunson | 2 | 8.50 | 0.14 | 2023-24 | 27 | 23.4 |

**1. Luka Dončić.** First on the client's metric and first on every finer version
of it. Five ballots in five seasons, more total vote share than anyone else in
the pool, and he was 24. He was also the best point guard in the league in the
last season the club can see: 33.9 points a game, a PER of 28.1, and the highest
assist percentage in the pool at 44.3. Price in the availability. He has not
played more than 70 games in any of the seven seasons on file.

**2. Shai Gilgeous-Alexander.** The metric ranks him fourth, on two appearances
against three. Every other reading puts him first or second: the best average
finish in the pool at 3.5, the second-highest vote share on half as many
appearances, and both of those appearances in the last two seasons of the window,
rising from fifth to second. He was 25, played 75 games and had the highest PER
of the thirteen. Two recent appearances by a 25-year-old are worth more than
three older ones by a 38-year-old, and the client's metric has no way of saying
so.

**3. Stephen Curry.** Second on the metric, and he survives the check. He did not
make the 2023-24 ballot, but a PER of 20.6 at 35 with 74 games played is a
starter comfortably above the league average, and only three of the thirteen shot
more efficiently that season. Buy him for one or two seasons rather than for a
rebuild, and expect the games played to fall.

**Not Chris Paul**, whom the metric ranks third. Three appearances between
2019-20 and 2021-22, none since, and by 2023-24 he was a 38-year-old scoring 9.2
points a game at a PER of 14.7 in 58 appearances. He is a useful veteran and the
wrong answer to "we want a strong start next season".

If the club would rather buy age and games than peak, **Jalen Brunson** is the
alternative to Curry. Two appearances, rising from twelfth to fifth, 27 years old,
and 77 games in 2023-24, the most of anyone in the pool.

What this rests on, and where it is thin:

- The pool is 13 players out of 200 point guards. The client's metric is a fame
  filter rather than an ability filter, and any point guard who was very good
  without being famous is missing from every list in this notebook.
- Ballot appearances measure what about a hundred voters thought, not what a
  player did. The whole exercise inherits whatever the voters got wrong.
- 2024-25 and 2025-26 are used only to grade the metric after the fact. Every
  name recommended above is justified from data the club had in the summer of
  2024.
- Nothing here knows whether a player is for sale, what he costs, or whether he
  would sign. Dončić was traded mid-season in 2024-25, which is a reminder that
  this is a market and not a shelf.
- This database has no wins or losses in it at all. "Increase its chances of
  success" cannot be measured directly, and PER, VORP and win shares are models
  of a player's contribution rather than observations of it.